# 01 — Perguntas para a análise exploratória de dados

Este notebook orienta a exploração de `Intrusion.csv`, conjunto principal da primeira análise do projeto MQTT Intrusion IDS. O objetivo desta etapa é compreender os dados e produzir evidências sobre qualidade, desbalanceamento, dependência temporal e risco de vazamento **antes** de limpar, selecionar features ou treinar modelos.

> As perguntas abaixo são hipóteses de investigação, não conclusões. Toda resposta deve distinguir evidência observada de interpretação.

## 2. Alvo e desbalanceamento

- Quais valores aparecem em `type` e qual deles define inequivocamente a classe positiva de intrusão?
- As contagens por classe confirmam 1.898 intrusões e a soma das classes coincide com o total de linhas?
- A prevalência global de intrusão (aproximadamente 2,35% na cópia conhecida) se mantém ao longo do tempo ou está concentrada em poucos intervalos?
- Existem rótulos ausentes, ambíguos, inconsistentes ou grafados de formas diferentes?
- Qual seria o desempenho trivial de um classificador que sempre prevê a classe majoritária e por que accuracy isolada seria enganosa?
- A raridade ocorre apenas por frame ou também por número e duração de eventos de intrusão?

## 3. Schema, tipos e qualidade estrutural

- Quais tipos foram inferidos e quais divergem do significado esperado de cada coluna?
- Há números armazenados como texto, categorias misturadas com sentinelas ou coerções que poderiam ocultar valores inválidos?
- Quais colunas são 100% vazias, constantes ou quase constantes? Elas indicam campos protocolares opcionais, falha de extração ou ausência real do fenômeno?
- Qual é a taxa de valores ausentes por coluna e por classe? A ausência em si pode carregar informação sobre o alvo?
- Existem padrões de ausência que sempre ocorrem juntos e revelam tipo de mensagem, equipamento ou cenário de captura?
- Quais valores violam domínios plausíveis, como portas, tamanhos, flags, QoS ou intervalos temporais?
- Toda coluna aparece exatamente uma vez no dicionário de dados, com tipo observado, significado, missingness, cardinalidade e política preliminar?

## 4. Duplicatas e dependência entre observações

- Existem exatamente 30 duplicatas de linha completas na versão conhecida? Quais classes elas reproduzem?
- As duplicatas são retransmissões legítimas do tráfego, repetição criada na extração ou cópias acidentais?
- A definição de duplicata muda quando se excluem número do frame, timestamp ou metadados de captura?
- Duplicatas ou quase duplicatas estão próximas no tempo ou aparecem em regiões distantes da captura?
- Que risco haveria se observações idênticas ou pertencentes à mesma rajada fossem separadas entre treino e validação?
- Qual regra de ordenação permite remover duplicatas preservando deterministicamente a primeira ocorrência em uma etapa futura, sem alterar o raw?

## 5. Distribuições e relações entre features

- Quais features numéricas apresentam assimetria, caudas longas, muitos zeros, valores extremos ou baixa variância?
- Os extremos representam tráfego válido, ataque, erro de medição ou codificação especial?
- Quais features categóricas têm alta cardinalidade ou categorias observadas uma única vez?
- Quais pares de colunas são redundantes, determinísticos ou diferentes representações da mesma informação?
- Como as distribuições mudam entre tráfego normal e intrusão, e essa diferença persiste em diferentes janelas temporais?
- As associações observadas têm significado protocolar plausível ou parecem artefatos do cenário de captura?
- Quais gráficos preservam a visibilidade da classe minoritária sem distorcer contagens ou esconder sobreposição?

## 6. Estrutura temporal e eventos

- Qual coluna, ou combinação de colunas, representa melhor o tempo real da observação? Ela é monotônica e tem resolução suficiente?
- Os ataques formam rajadas contíguas? Quantos eventos existem e qual é a distribuição de duração e espaçamento entre eles?
- Há mudanças de prevalência, volume, missingness ou distribuição das features ao longo da captura?
- Tráfego normal antes, durante e depois de um ataque tem o mesmo perfil?
- Frames vizinhos são semelhantes o bastante para invalidar uma divisão aleatória convencional?
- Blocos temporais de 10, 30 e 60 segundos contêm ambas as classes em quantidade suficiente para agrupamento e validação?
- Os últimos 30% no tempo diferem dos primeiros 70% em classe, identidade ou distribuição de features, sugerindo drift no futuro holdout?
- Que visualização — linha temporal, heatmap por janela ou duração de eventos — evidencia melhor a concentração dos ataques?

## 7. Vazamento por identidade, conteúdo e cenário

- IPs, MACs, client IDs, tópicos ou payloads aparecem exclusivamente, ou quase exclusivamente, em uma classe?
- Uma tabela cruzada quase perfeita entre identificador e alvo representa comportamento generalizável ou apenas a identidade dos equipamentos usados na coleta?
- Quantas identidades ou categorias presentes em períodos anteriores reaparecem nos períodos posteriores? Há categorias inéditas no fim da captura?
- O alvo pode ser inferido diretamente do número do frame, timestamp absoluto, arquivo, sequência de captura ou outro metadado?
- Payload ou tópico contém texto que nomeia o ataque, a ferramenta ou o cenário experimental?
- Uma feature aparentemente técnica é proxy determinístico de IP, dispositivo, tempo ou rótulo?
- Quanto da associação com o alvo desaparece quando os dados são analisados por diferentes janelas temporais ou identidades?
- Quais evidências justificam excluir uma coluna da política principal e quais justificam reservá-la apenas para ablação diagnóstica?

## 8. Portabilidade e política de features

- A feature estaria disponível em tempo de inferência sem conhecer o futuro ou o rótulo?
- A feature existe em outro broker, dispositivo, rede e janela de captura, ou codifica uma identidade específica deste experimento?
- Quais colunas devem ser classificadas como `portable`, `excluded`, `derivable` ou `ablation_only`, e qual evidência sustenta cada escolha?
- É possível substituir identificadores crus por derivações mínimas e generalizáveis, como direção privado/público, flag de porta MQTT, presença de campo ou quantidade de campos preenchidos?
- Cada derivação proposta preserva sinal protocolar sem reconstruir silenciosamente a identidade ou a ordem absoluta?
- Quais features dependem de estado, agregação em janela ou inspeção de payload e, portanto, mudam o custo e a viabilidade operacional do IDS?
- Que informação deverá permanecer apenas no manifesto para auditoria, sem entrar na matriz de features?

## 9. Consequências para validação e métricas

- Que evidências da EDA sustentam reservar os últimos 30% no tempo como holdout, em vez de usar uma divisão aleatória?
- A granularidade de 30 segundos separa adequadamente grupos dependentes sem tornar os folds inviáveis?
- Há grupos temporais contendo somente uma classe e como isso afeta a estratificação agrupada?
- A análise por frame superestima utilidade quando muitos frames pertencem ao mesmo evento?
- Além de Macro-F1 e recall de intrusão, quais evidências descritivas antecipam a necessidade de PR-AUC, FPR e falsos alertas por 10.000 frames normais?
- Que padrão de eventos permitiria medir tempo até o primeiro alerta de forma inequívoca?
- Quais achados devem virar invariantes testáveis do contrato de dados, e quais devem permanecer hipóteses sujeitas a drift?

## 10. Síntese esperada

Ao concluir a exploração, deve ser possível responder: 

1. Qual é a unidade observada e quão confiáveis são schema, tipos e rótulos?
2. Onde estão os principais problemas de qualidade e dependência entre linhas?
3. Como as intrusões se distribuem por classe, tempo e evento?
4. Quais associações são plausivelmente portáveis e quais indicam vazamento de identidade, conteúdo, ordem ou cenário?
5. Que colunas podem entrar na política principal, quais devem ser excluídas, derivadas ou usadas somente em ablações?
6. Quais resultados precisam se tornar contratos automatizados e quais decisões novas precisam de ADR?

> Esta etapa termina com evidências, dicionário de dados e decisões documentadas. Treinamento, seleção de features e consulta orientada a modelos do holdout ficam fora do escopo deste notebook.

In [8]:
import os 
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import kagglehub
from dotenv import load_dotenv

load_dotenv()

handle = os.environ.get("KAGGLE_DATASET_VERSION_HANDLE")
dataset_dir = Path(kagglehub.dataset_download(handle))

intrusion = dataset_dir / "Intrusion.csv"
dos = dataset_dir / "DoS.csv"
mitm = dataset_dir / "MitM.csv"

# conferindo que os caminhos foram importados corretamente 

df_intrusion = pd.read_csv(intrusion)
df_intrusion.head()


/tmp/ipykernel_123891/647220643.py:26: DtypeWarning: Columns (0: mqtt.clientid, 1: mqtt.conack.flags, 2: mqtt.conflags, 3: mqtt.protoname, 4: mqtt.topic) have mixed types. Specify dtype option on import or set low_memory=False.
  df_intrusion = pd.read_csv(intrusion)


,frame.time_delta,frame.time_delta_displayed,frame.time_epoch,frame.time_invalid,frame.time_relative,ip.src,ip.dst,tcp.srcport,tcp.dstport,eth.src,...,mqtt.topic,mqtt.topic_len,mqtt.username,mqtt.username_len,mqtt.ver,mqtt.willmsg,mqtt.willmsg_len,mqtt.willtopic,mqtt.willtopic_len,type
0,0.000000,0.000000,1.535972e+09,NaN,0.000000,192.168.1.227,31.13.83.170,57188.0,443.0,48:5a:3f:93:39:9c,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal
1,0.005887,0.005887,1.535972e+09,NaN,0.005887,31.13.83.170,192.168.1.227,443.0,57188.0,18:a6:f7:eb:77:26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal
2,0.001288,0.001288,1.535972e+09,NaN,0.007175,192.168.1.227,31.13.83.170,57188.0,443.0,48:5a:3f:93:39:9c,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal
3,0.010402,0.010402,1.535972e+09,NaN,0.017577,192.168.1.227,31.13.83.170,57188.0,443.0,48:5a:3f:93:39:9c,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal
4,0.004078,0.004078,1.535972e+09,NaN,0.021655,192.168.1.227,31.13.83.170,57188.0,443.0,48:5a:3f:93:39:9c,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal


In [9]:
df_dos = pd.read_csv(dos)
df_dos.head()

/tmp/ipykernel_123891/2744016784.py:1: DtypeWarning: Columns (0: mqtt.clientid, 1: mqtt.conack.flags, 2: mqtt.conflags, 3: mqtt.msg, 4: mqtt.protoname) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dos = pd.read_csv(dos)


,frame.time_delta,frame.time_delta_displayed,frame.time_epoch,frame.time_invalid,frame.time_relative,ip.src,ip.dst,tcp.srcport,tcp.dstport,eth.src,...,mqtt.topic,mqtt.topic_len,mqtt.username,mqtt.username_len,mqtt.ver,mqtt.willmsg,mqtt.willmsg_len,mqtt.willtopic,mqtt.willtopic_len,type
0,0.000000,0.000000,1.522235e+09,NaN,0.000000,192.168.1.1,192.168.1.196,22.0,51749.0,18:a6:f7:eb:77:26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal
1,0.001922,0.001922,1.522235e+09,NaN,0.001922,192.168.1.1,192.168.1.196,22.0,51749.0,18:a6:f7:eb:77:26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal
2,0.001010,0.001010,1.522235e+09,NaN,0.002932,192.168.1.196,192.168.1.1,51749.0,22.0,30:5a:3a:62:72:80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal
3,0.008597,0.008597,1.522235e+09,NaN,0.011529,192.168.1.1,192.168.1.196,22.0,51749.0,18:a6:f7:eb:77:26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal
4,0.002925,0.002925,1.522235e+09,NaN,0.014454,192.168.1.196,192.168.1.1,51749.0,22.0,30:5a:3a:62:72:80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal


In [10]:
df_mitm = pd.read_csv(mitm)
df_mitm.head()

/tmp/ipykernel_123891/2693207988.py:1: DtypeWarning: Columns (0: mqtt.clientid, 1: mqtt.conack.flags, 2: mqtt.conflags, 3: mqtt.protoname, 4: mqtt.topic) have mixed types. Specify dtype option on import or set low_memory=False.
  df_mitm = pd.read_csv(mitm)


,frame.time_delta,frame.time_delta_displayed,frame.time_epoch,frame.time_invalid,frame.time_relative,ip.src,ip.dst,tcp.srcport,tcp.dstport,eth.src,...,mqtt.topic,mqtt.topic_len,mqtt.username,mqtt.username_len,mqtt.ver,mqtt.willmsg,mqtt.willmsg_len,mqtt.willtopic,mqtt.willtopic_len,type
0,0.000000,0.000000,1.529668e+09,NaN,0.000000,192.168.1.1,192.168.1.102,22.0,53940.0,18:a6:f7:eb:77:26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal
1,0.000481,0.000481,1.529668e+09,NaN,0.000481,192.168.1.1,192.168.1.102,22.0,53940.0,18:a6:f7:eb:77:26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal
2,0.000423,0.000423,1.529668e+09,NaN,0.000904,192.168.1.1,192.168.1.102,22.0,53940.0,18:a6:f7:eb:77:26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal
3,0.000274,0.000274,1.529668e+09,NaN,0.001178,192.168.1.102,192.168.1.1,53940.0,22.0,30:5a:3a:62:72:80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal
4,0.000861,0.000861,1.529668e+09,NaN,0.002039,192.168.1.102,192.168.1.1,53940.0,22.0,30:5a:3a:62:72:80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal


In [12]:
print(df_dos.info())
print('\n')


<class 'pandas.DataFrame'>
RangeIndex: 94625 entries, 0 to 94624
Data columns (total 67 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   frame.time_delta            94625 non-null  float64
 1   frame.time_delta_displayed  94625 non-null  float64
 2   frame.time_epoch            94625 non-null  float64
 3   frame.time_invalid          0 non-null      float64
 4   frame.time_relative         94625 non-null  float64
 5   ip.src                      93592 non-null  str    
 6   ip.dst                      93592 non-null  str    
 7   tcp.srcport                 88460 non-null  float64
 8   tcp.dstport                 88460 non-null  float64
 9   eth.src                     94625 non-null  str    
 10  eth.dst                     94625 non-null  str    
 11  frame.cap_len               94625 non-null  int64  
 12  frame.coloring_rule.name    0 non-null      float64
 13  frame.coloring_rule.string  0 non-null    

In [13]:
print(df_mitm.info())
print('\n')


<class 'pandas.DataFrame'>
RangeIndex: 110668 entries, 0 to 110667
Data columns (total 67 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   frame.time_delta            110668 non-null  float64
 1   frame.time_delta_displayed  110668 non-null  float64
 2   frame.time_epoch            110668 non-null  float64
 3   frame.time_invalid          0 non-null       float64
 4   frame.time_relative         110668 non-null  float64
 5   ip.src                      102435 non-null  str    
 6   ip.dst                      102435 non-null  str    
 7   tcp.srcport                 99514 non-null   float64
 8   tcp.dstport                 99514 non-null   float64
 9   eth.src                     110668 non-null  str    
 10  eth.dst                     110668 non-null  str    
 11  frame.cap_len               110668 non-null  int64  
 12  frame.coloring_rule.name    0 non-null       float64
 13  frame.coloring_rule.strin

In [14]:
print(df_intrusion.info())
print('\n')

<class 'pandas.DataFrame'>
RangeIndex: 80893 entries, 0 to 80892
Data columns (total 67 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   frame.time_delta            80893 non-null  float64
 1   frame.time_delta_displayed  80893 non-null  float64
 2   frame.time_epoch            80893 non-null  float64
 3   frame.time_invalid          0 non-null      float64
 4   frame.time_relative         80893 non-null  float64
 5   ip.src                      77106 non-null  str    
 6   ip.dst                      77106 non-null  str    
 7   tcp.srcport                 71757 non-null  float64
 8   tcp.dstport                 71757 non-null  float64
 9   eth.src                     80893 non-null  str    
 10  eth.dst                     80893 non-null  str    
 11  frame.cap_len               80893 non-null  int64  
 12  frame.coloring_rule.name    0 non-null      float64
 13  frame.coloring_rule.string  0 non-null    

In [16]:
column_summary = (
      pd.DataFrame({
          "dtype": df_intrusion.dtypes,
          "missing_count": df_intrusion.isna().sum(),
          "missing_percentage": df_intrusion.isna().mean().mul(100),
          "unique_count": df_intrusion.nunique(dropna=True),
      })
      .sort_values("missing_percentage", ascending=False)
  )
column_summary

,dtype,missing_count,missing_percentage,unique_count
frame.coloring_rule.string,float64,80893,100.0,0
frame.comment,float64,80893,100.0,0
frame.time_invalid,float64,80893,100.0,0
mqtt.willmsg_len,float64,80893,100.0,0
mqtt.willmsg,float64,80893,100.0,0
...,...,...,...,...
frame.number,int64,0,0.0,80863
eth.dst,str,0,0.0,24
frame.cap_len,int64,0,0.0,1356
frame.encap_type,int64,0,0.0,1


In [18]:
missing_summary = (
    pd.DataFrame({
        "missing_count": df_intrusion.isna().sum(),
        "missing_percentage": df_intrusion.isna().mean().mul(100),
    })
    .sort_values("missing_percentage", ascending=False)
)

with pd.option_context("display.max_rows", None):
    display(missing_summary)

,missing_count,missing_percentage
frame.coloring_rule.string,80893,100.000000
frame.comment,80893,100.000000
frame.time_invalid,80893,100.000000
mqtt.willmsg_len,80893,100.000000
mqtt.willmsg,80893,100.000000
mqtt.willtopic,80893,100.000000
frame.file_off,80893,100.000000
frame.comment.expert,80893,100.000000
frame.coloring_rule.name,80893,100.000000
frame.interface_id,80893,100.000000
